In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Snake RL Generalization Study v3
**Resilient notebook** — each experiment cell caches results to `results_cache.pkl`.
Rerunning the notebook skips completed experiments automatically.

Key changes from v2:
- DQN: 150k steps, larger buffer (100k), slower ε-decay, lr=5e-4 → less variance
- PPO: 800k steps → keeps climbing
- All plots: line plots with CI bands, distinct palette
- Plots saved to `snake_results/`

In [2]:
# ── Cell 0: Imports & Config ──────────────────────────────────────────────────
import numpy as np
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import deque
from itertools import product as iproduct
import warnings, os, math, pickle
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

os.makedirs('/content/drive/MyDrive/snake_results', exist_ok=True)
CACHE_FILE = '/content/drive/MyDrive/snake_rl_cache.pkl'

# ── Global hypers ─────────────────────────────────────────────────────────────
MAX_STEPS_K    = 3
SEEDS          = 3
DQN_STEPS      = 150_000   # env steps per DQN run (plateau ~100k)
PPO_STEPS      = 800_000   # env steps per PPO run
N_ENVS_DQN     = 8
N_ENVS_PPO     = 16
EVAL_EPISODES  = 50
GRID_SIZES     = [6, 8, 9, 10, 12]
OBS_COUNTS     = [0, 2, 4, 6, 8]
TRAIN_SIZE     = 9
TRAIN_OBS      = 0
EVAL_CFGS      = [(s, 0) for s in GRID_SIZES]
OBS_CFGS       = [(TRAIN_SIZE, o) for o in OBS_COUNTS]
TRAIN_CFG      = (TRAIN_SIZE, TRAIN_OBS)
REPS           = ['local', 'extended', 'hybrid']

# ── Cache helpers ─────────────────────────────────────────────────────────────
def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'rb') as f:
            return pickle.load(f)
    return {}

def save_cache(cache):
    with open(CACHE_FILE, 'wb') as f:
        pickle.dump(cache, f)
    print(f'  [cache saved → {CACHE_FILE}]')

def cached(key, fn):
    """Run fn() only if key not already in cache. Returns result."""
    cache = load_cache()
    if key in cache:
        print(f'  [SKIP] "{key}" already cached.')
        return cache[key]
    print(f'  [RUN]  "{key}" ...')
    result = fn()
    cache[key] = result
    save_cache(cache)
    return result

print('Config loaded. Cache file:', CACHE_FILE)

Using device: cuda
Config loaded. Cache file: /content/drive/MyDrive/snake_rl_cache.pkl


In [3]:
# ── Cell 1: Environment ───────────────────────────────────────────────────────

class SnakeEnv:
    DIRECTIONS = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    STATE_DIMS = {'local': 11, 'extended': 19, 'hybrid': 26}

    def __init__(self, size=9, obstacle_count=0, rep='local'):
        self.size           = size
        self.obstacle_count = obstacle_count
        self.rep            = rep
        self.max_steps      = size * size * MAX_STEPS_K
        self._step_count    = 0
        self.reset()

    def reset(self):
        self.snake       = [(self.size // 2, self.size // 2)]
        self.direction   = (0, 1)
        self._step_count = 0
        self._generate_obstacles()
        self._spawn_food()
        return self.get_state()

    def _generate_obstacles(self):
        self.obstacles = set()
        if self.obstacle_count == 0:
            return
        forbidden = set(self.snake)
        attempts  = 0
        while len(self.obstacles) < self.obstacle_count and attempts < 1000:
            r = random.randint(0, self.size - 1)
            c = random.randint(0, self.size - 1)
            if (r, c) not in forbidden:
                self.obstacles.add((r, c))
                forbidden.add((r, c))
            attempts += 1

    def _spawn_food(self):
        occupied = set(self.snake) | self.obstacles
        free = [(r, c) for r in range(self.size)
                        for c in range(self.size)
                        if (r, c) not in occupied]
        self.food = random.choice(free) if free else self.snake[0]

    def is_collision(self, point):
        r, c = point
        return (
            point in self.obstacles
            or tuple(point) in [tuple(s) for s in self.snake]
            or not (0 <= r < self.size)
            or not (0 <= c < self.size)
        )

    def _snake_set(self):
        return set(self.snake)

    def _is_col_fast(self, point, snake_set):
        r, c = point
        return (
            point in snake_set
            or point in self.obstacles
            or not (0 <= r < self.size)
            or not (0 <= c < self.size)
        )

    def get_state(self):
        if self.rep == 'local':    return self._state_local()
        if self.rep == 'extended': return self._state_extended()
        if self.rep == 'hybrid':   return self._state_hybrid()
        raise ValueError(f'Unknown rep: {self.rep}')

    def _state_local(self):
        head = self.snake[0]
        dr, dc = self.direction
        ss = self._snake_set()
        straight = (head[0] + dr,  head[1] + dc)
        right_p  = (head[0] + dc,  head[1] - dr)
        left_p   = (head[0] - dc,  head[1] + dr)
        state = [
            int(self._is_col_fast(straight, ss)),
            int(self._is_col_fast(right_p,  ss)),
            int(self._is_col_fast(left_p,   ss)),
            int(self.food[1] < head[1]),
            int(self.food[1] > head[1]),
            int(self.food[0] < head[0]),
            int(self.food[0] > head[0]),
            int(self.direction == (-1, 0)),
            int(self.direction == (1,  0)),
            int(self.direction == (0, -1)),
            int(self.direction == (0,  1)),
        ]
        return np.array(state, dtype=np.float32)

    def _state_extended(self):
        local = self._state_local()
        head  = self.snake[0]
        ss    = self._snake_set()
        ray_dirs = [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]
        raycasts = []
        for rd, rc in ray_dirs:
            dist = 0
            r, c = head
            while True:
                r += rd; c += rc; dist += 1
                if not (0 <= r < self.size and 0 <= c < self.size): break
                if (r, c) in ss or (r, c) in self.obstacles: break
            raycasts.append(dist / self.size)
        return np.concatenate([local, raycasts]).astype(np.float32)

    def _state_hybrid(self):
        extended = self._state_extended()
        head     = self.snake[0]
        body_frac = len(self.snake) / self.size ** 2
        obs_frac  = len(self.obstacles) / self.size ** 2
        food_dist = (abs(head[0] - self.food[0]) +
                     abs(head[1] - self.food[1])) / (2 * self.size)
        mid = self.size // 2
        quad = [0, 0, 0, 0]
        for r, c in self.snake:
            qi = (0 if r < mid else 2) + (0 if c < mid else 1)
            quad[qi] += 1
        quad = [q / max(len(self.snake), 1) for q in quad]
        global_feats = np.array([body_frac, obs_frac, food_dist] + quad, dtype=np.float32)
        return np.concatenate([extended, global_feats]).astype(np.float32)

    def step(self, action):
        self._step_count += 1
        idx = self.DIRECTIONS.index(self.direction)
        if action == 1:   idx = (idx + 1) % 4
        elif action == 2: idx = (idx - 1) % 4
        self.direction = self.DIRECTIONS[idx]
        head     = self.snake[0]
        new_head = (head[0] + self.direction[0], head[1] + self.direction[1])
        ss = self._snake_set()
        if self._is_col_fast(new_head, ss):
            return self.get_state(), -10.0, True, False, False
        self.snake.insert(0, new_head)
        old_dist = abs(head[0] - self.food[0]) + abs(head[1] - self.food[1])
        new_dist = abs(new_head[0] - self.food[0]) + abs(new_head[1] - self.food[1])
        reward   = float(old_dist - new_dist) - 0.05
        ate_food = (new_head == self.food)
        if ate_food:
            reward += 10.0
            self._spawn_food()
        else:
            self.snake.pop()
        truncated = (self._step_count >= self.max_steps)
        return self.get_state(), reward, False, truncated, ate_food

print('SnakeEnv defined.')

SnakeEnv defined.


In [4]:
# ── Cell 2: Vectorised Env + Reward Normaliser ────────────────────────────────

class DummyVecEnv:
    def __init__(self, env_fns):
        self.envs      = [fn() for fn in env_fns]
        self.n_envs    = len(self.envs)
        self.state_dim = self.envs[0].reset().shape[0]

    def reset(self):
        return np.array([env.reset() for env in self.envs], dtype=np.float32)

    def step(self, actions):
        obs_list, rew_list, done_list, trunc_list, info_list = [], [], [], [], []
        for env, action in zip(self.envs, actions):
            obs, rew, done, trunc, ate = env.step(int(action))
            if done or trunc:
                obs = env.reset()
            obs_list.append(obs);  rew_list.append(rew)
            done_list.append(done); trunc_list.append(trunc)
            info_list.append({'ate_food': ate, 'done': done, 'trunc': trunc})
        return (
            np.array(obs_list,   dtype=np.float32),
            np.array(rew_list,   dtype=np.float32),
            np.array(done_list,  dtype=bool),
            np.array(trunc_list, dtype=bool),
            info_list,
        )

    def close(self): pass


def make_vec_env(n_envs, size, obs_count, rep):
    return DummyVecEnv([
        lambda s=size, o=obs_count, r=rep: SnakeEnv(s, o, r)
        for _ in range(n_envs)
    ])


class RunningMeanStd:
    def __init__(self, eps=1e-4):
        self.mean = 0.0; self.var = 1.0; self.count = eps

    def update(self, x):
        batch_mean = float(np.mean(x)); batch_var = float(np.var(x)); batch_n = len(x)
        total = self.count + batch_n; delta = batch_mean - self.mean
        self.mean  += delta * batch_n / total
        m_a = self.var * self.count; m_b = batch_var * batch_n
        self.var    = (m_a + m_b + delta**2 * self.count * batch_n / total) / total
        self.count  = total

    def normalize(self, x, clip=10.0):
        return np.clip((x - self.mean) / (np.sqrt(self.var) + 1e-8), -clip, clip)

print('VecEnv + RunningMeanStd defined.')

VecEnv + RunningMeanStd defined.


In [5]:
# ── Cell 3: Neural Networks ───────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, input_dim, output_dim, hidden=(256, 128)):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)
        # He init for ReLU networks
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                nn.init.constant_(m.bias, 0.0)

    def forward(self, x):
        return self.net(x)


class ActorCritic(nn.Module):
    def __init__(self, input_dim, n_actions, hidden=(256, 128)):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden:
            layers += [nn.Linear(prev, h), nn.Tanh()]
            prev = h
        self.trunk  = nn.Sequential(*layers)
        self.actor  = nn.Linear(prev, n_actions)
        self.critic = nn.Linear(prev, 1)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.orthogonal_(m.weight, gain=np.sqrt(2))
                nn.init.constant_(m.bias, 0.0)
        nn.init.orthogonal_(self.actor.weight, gain=0.01)
        nn.init.orthogonal_(self.critic.weight, gain=1.0)

    def forward(self, x):
        h = self.trunk(x)
        return self.actor(h), self.critic(h)

    def evaluate(self, x, actions):
        logits, values = self(x)
        dist    = Categorical(logits=logits)
        log_p   = dist.log_prob(actions)
        entropy = dist.entropy()
        return log_p, entropy, values.squeeze(-1)

    def act(self, x):
        with torch.no_grad():
            logits, values = self(x)
        dist    = Categorical(logits=logits)
        actions = dist.sample()
        log_p   = dist.log_prob(actions)
        return actions, log_p, values.squeeze(-1)

    def act_greedy(self, x):
        with torch.no_grad():
            logits, _ = self(x)
        return logits.argmax(-1)

print('Neural networks defined.')

Neural networks defined.


In [6]:
# ── Cell 4: DQN Agent (lower variance) ───────────────────────────────────────

class ReplayBuffer:
    def __init__(self, capacity, state_dim):
        self.capacity = capacity; self.ptr = 0; self.size = 0
        self.states  = np.zeros((capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros((capacity,),           dtype=np.int64)
        self.rewards = np.zeros((capacity,),           dtype=np.float32)
        self.next_s  = np.zeros((capacity, state_dim), dtype=np.float32)
        self.dones   = np.zeros((capacity,),           dtype=np.float32)

    def push_batch(self, s, a, r, ns, d):
        n    = len(r)
        idxs = np.arange(self.ptr, self.ptr + n) % self.capacity
        self.states[idxs]  = s;  self.actions[idxs] = a
        self.rewards[idxs] = r;  self.next_s[idxs]  = ns
        self.dones[idxs]   = d.astype(np.float32)
        self.ptr  = (self.ptr + n) % self.capacity
        self.size = min(self.size + n, self.capacity)

    def sample(self, k):
        idxs = np.random.randint(0, self.size, k)
        return (
            torch.FloatTensor(self.states[idxs]).to(DEVICE),
            torch.LongTensor(self.actions[idxs]).to(DEVICE),
            torch.FloatTensor(self.rewards[idxs]).to(DEVICE),
            torch.FloatTensor(self.next_s[idxs]).to(DEVICE),
            torch.FloatTensor(self.dones[idxs]).to(DEVICE),
        )

    def __len__(self): return self.size


class DQNAgent:
    """
    DQN / Double-DQN — tuned for low variance:
    - Larger network (256,128)
    - Larger buffer (100k), slower ε-decay (0.9998), lower lr (5e-4)
    - Larger warmup (2000), tighter grad clip (5)
    - Target sync every 200 grad steps (was 100)
    """
    def __init__(self, state_dim, n_actions=3, double=False,
                 lr=5e-4, gamma=0.99,
                 eps_start=1.0, eps_end=0.05, eps_decay=0.9998,
                 batch_size=256, target_update=200,
                 buffer_size=100_000, warmup=2000):
        self.state_dim     = state_dim
        self.n_actions     = n_actions
        self.double        = double
        self.gamma         = gamma
        self.eps           = eps_start
        self.eps_end       = eps_end
        self.eps_decay     = eps_decay
        self.batch_size    = batch_size
        self.target_update = target_update
        self.warmup        = warmup
        self.grad_steps    = 0

        self.policy_net = MLP(state_dim, n_actions).to(DEVICE)
        self.target_net = MLP(state_dim, n_actions).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer  = optim.Adam(self.policy_net.parameters(), lr=lr)
        self.buffer     = ReplayBuffer(buffer_size, state_dim)

    def select_actions(self, states_np, greedy=False):
        n = states_np.shape[0]
        if not greedy and random.random() < self.eps:
            return np.random.randint(0, self.n_actions, n)
        with torch.no_grad():
            t = torch.FloatTensor(states_np).to(DEVICE)
            return self.policy_net(t).argmax(1).cpu().numpy()

    def learn(self):
        if len(self.buffer) < self.warmup:
            return None
        s, a, r, ns, d = self.buffer.sample(self.batch_size)
        q_vals = self.policy_net(s).gather(1, a.unsqueeze(1))
        with torch.no_grad():
            if self.double:
                best_a = self.policy_net(ns).argmax(1, keepdim=True)
                next_q = self.target_net(ns).gather(1, best_a)
            else:
                next_q = self.target_net(ns).max(1, keepdim=True)[0]
            target = (r + (1 - d) * self.gamma * next_q.squeeze()).unsqueeze(1)
        loss = F.smooth_l1_loss(q_vals, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 5.0)
        self.optimizer.step()
        self.grad_steps += 1
        self.eps = max(self.eps_end, self.eps * self.eps_decay)
        if self.grad_steps % self.target_update == 0:
            self.target_net.load_state_dict(self.policy_net.state_dict())
        return loss.item()

print('DQNAgent defined.')

DQNAgent defined.


In [7]:
# ── Cell 5: PPO Agent ─────────────────────────────────────────────────────────

class PPOAgent:
    def __init__(self, state_dim, n_actions=3, n_envs=16,
                 lr=1e-4, gamma=0.99, lam=0.95,
                 clip_eps=0.2, entropy_coef=0.03, vf_coef=0.5,
                 epochs=10, rollout_len=1024, mini_batch=256,
                 normalize_rewards=True):
        self.n_envs         = n_envs
        self.gamma          = gamma
        self.lam            = lam
        self.clip_eps       = clip_eps
        self.entropy_coef   = entropy_coef
        self.vf_coef        = vf_coef
        self.epochs         = epochs
        self.rollout_len    = rollout_len
        self.mini_batch     = mini_batch
        self.normalize_rews = normalize_rewards
        self.ac        = ActorCritic(state_dim, n_actions).to(DEVICE)
        self.optimizer = optim.Adam(self.ac.parameters(), lr=lr, eps=1e-5)
        self.rew_rms   = RunningMeanStd()
        T, E = rollout_len, n_envs
        self.buf_obs   = np.zeros((T, E, state_dim), dtype=np.float32)
        self.buf_acts  = np.zeros((T, E), dtype=np.int64)
        self.buf_rews  = np.zeros((T, E), dtype=np.float32)
        self.buf_vals  = np.zeros((T, E), dtype=np.float32)
        self.buf_logp  = np.zeros((T, E), dtype=np.float32)
        self.buf_dones = np.zeros((T, E), dtype=np.float32)
        self.buf_trunc = np.zeros((T, E), dtype=np.float32)

    def collect_rollout(self, vec_env, obs):
        food_counts = np.zeros(self.n_envs, dtype=np.int32)
        ep_food_log = []
        for t in range(self.rollout_len):
            obs_t = torch.FloatTensor(obs).to(DEVICE)
            acts, logp, vals = self.ac.act(obs_t)
            acts_np = acts.cpu().numpy()
            next_obs, rews, dones, truncs, infos = vec_env.step(acts_np)
            for i, info in enumerate(infos):
                if info['ate_food']:  food_counts[i] += 1
                if info['done'] or info['trunc']:
                    ep_food_log.append(food_counts[i]); food_counts[i] = 0
            self.buf_obs[t]   = obs
            self.buf_acts[t]  = acts_np
            self.buf_rews[t]  = rews
            self.buf_vals[t]  = vals.cpu().numpy()
            self.buf_logp[t]  = logp.cpu().numpy()
            self.buf_dones[t] = dones.astype(np.float32)
            self.buf_trunc[t] = truncs.astype(np.float32)
            obs = next_obs
        return obs, ep_food_log

    def compute_gae(self, last_obs, last_dones, last_truncs):
        with torch.no_grad():
            last_t  = torch.FloatTensor(last_obs).to(DEVICE)
            _, last_v = self.ac(last_t)
            last_v  = last_v.squeeze(-1).cpu().numpy()
        T, E = self.rollout_len, self.n_envs
        advantages = np.zeros((T, E), dtype=np.float32)
        gae        = np.zeros(E, dtype=np.float32)
        for t in reversed(range(T)):
            if t == T - 1:
                next_val  = last_v * (1 - last_dones.astype(np.float32))
                next_val += last_v * last_truncs.astype(np.float32)
            else:
                next_non_term = 1.0 - self.buf_dones[t + 1]
                next_val      = self.buf_vals[t + 1] * next_non_term
            rews = self.buf_rews[t]
            if self.normalize_rews:
                self.rew_rms.update(rews.flatten())
                rews = self.rew_rms.normalize(rews)
            delta         = rews + self.gamma * next_val - self.buf_vals[t]
            non_term      = 1.0 - self.buf_dones[t]
            gae           = delta + self.gamma * self.lam * non_term * gae
            advantages[t] = gae
        returns = advantages + self.buf_vals
        return advantages, returns

    def update(self, advantages, returns):
        T, E    = self.rollout_len, self.n_envs
        n_total = T * E
        obs_flat  = self.buf_obs.reshape(n_total, -1)
        acts_flat = self.buf_acts.reshape(n_total)
        logp_flat = self.buf_logp.reshape(n_total)
        adv_flat  = advantages.reshape(n_total)
        ret_flat  = returns.reshape(n_total)
        adv_flat  = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)
        obs_t  = torch.FloatTensor(obs_flat).to(DEVICE)
        acts_t = torch.LongTensor(acts_flat).to(DEVICE)
        logp_t = torch.FloatTensor(logp_flat).to(DEVICE)
        adv_t  = torch.FloatTensor(adv_flat).to(DEVICE)
        ret_t  = torch.FloatTensor(ret_flat).to(DEVICE)
        for _ in range(self.epochs):
            perm = np.random.permutation(n_total)
            for start in range(0, n_total, self.mini_batch):
                mb = perm[start: start + self.mini_batch]
                new_logp, entropy, vals = self.ac.evaluate(obs_t[mb], acts_t[mb])
                ratio  = (new_logp - logp_t[mb]).exp()
                surr1  = ratio * adv_t[mb]
                surr2  = ratio.clamp(1 - self.clip_eps, 1 + self.clip_eps) * adv_t[mb]
                a_loss = -torch.min(surr1, surr2).mean()
                v_loss = F.mse_loss(vals, ret_t[mb])
                loss   = a_loss + self.vf_coef * v_loss - self.entropy_coef * entropy.mean()
                self.optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.ac.parameters(), 0.5)
                self.optimizer.step()

print('PPOAgent defined.')

PPOAgent defined.


In [8]:
# ── Cell 6: Hamiltonian Baseline + Training Loops ─────────────────────────────

import heapq

def build_hamiltonian_cycle_prims(n):
    assert n % 2 == 0
    half = n // 2
    can_right = [[False]*half for _ in range(half)]
    can_down  = [[False]*half for _ in range(half)]
    visited   = [[False]*half for _ in range(half)]
    visited[0][0] = True
    frontier = []

    def add_frontier(r, c):
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < half and 0 <= nc < half and not visited[nr][nc]:
                heapq.heappush(frontier, (random.random(), r, c, nr, nc))

    add_frontier(0, 0)
    while frontier:
        _, pr, pc, r, c = heapq.heappop(frontier)
        if visited[r][c]: continue
        visited[r][c] = True
        if   r == pr+1: can_down[pr][pc]  = True
        elif r == pr-1: can_down[r][c]    = True
        elif c == pc+1: can_right[pr][pc] = True
        else:           can_right[r][c]   = True
        add_frontier(r, c)

    adj = [[set() for _ in range(n)] for _ in range(n)]
    def connect(r1,c1,r2,c2):
        adj[r1][c1].add((r2,c2)); adj[r2][c2].add((r1,c1))

    for sr in range(half):
        for sc in range(half):
            r, c = sr*2, sc*2
            if can_right[sr][sc] and sc+1 < half:
                connect(r,   c+1, r,   c+2)
                connect(r+1, c+1, r+1, c+2)
            if can_down[sr][sc] and sr+1 < half:
                connect(r+1, c,   r+2, c)
                connect(r+1, c+1, r+2, c+1)
            if not (sc > 0 and can_right[sr][sc-1]):         connect(r,   c,   r+1, c)
            if not (can_right[sr][sc] and sc+1 < half):      connect(r,   c+1, r+1, c+1)
            if not (sr > 0 and can_down[sr-1][sc]):          connect(r,   c,   r,   c+1)
            if not (can_down[sr][sc] and sr+1 < half):       connect(r+1, c,   r+1, c+1)

    path = []; cur, prev = (0,0), None
    for _ in range(n*n):
        path.append(cur)
        nxt = [nb for nb in adj[cur[0]][cur[1]] if nb != prev]
        if not nxt: break
        prev, cur = cur, nxt[0]
    return path


_ham_cache = {}
def get_ham_cycle(n):
    if n not in _ham_cache:
        _ham_cache[n] = build_hamiltonian_cycle_prims(n)
    return _ham_cache[n]


class HamiltonianAgent:
    def __init__(self, size):
        self.size    = size
        self.path    = get_ham_cycle(size)
        self.idx_map = {cell: i for i, cell in enumerate(self.path)}
        self._step   = 0

    def reset(self, env):
        self._step = self.idx_map.get(env.snake[0], 0)

    def act(self, env):
        head       = env.snake[0]
        self._step = self.idx_map.get(head, self._step)
        next_idx   = (self._step + 1) % len(self.path)
        next_cell  = self.path[next_idx]
        dr, dc     = next_cell[0]-head[0], next_cell[1]-head[1]
        target_dir = (dr, dc)
        DIRECTIONS = env.DIRECTIONS
        if target_dir not in DIRECTIONS:
            base_idx = DIRECTIONS.index(env.direction)
            for candidate in [0, 1, 2]:
                idx = base_idx
                if candidate == 1: idx = (idx+1) % 4
                if candidate == 2: idx = (idx-1) % 4
                nxt = (head[0]+DIRECTIONS[idx][0], head[1]+DIRECTIONS[idx][1])
                if not env.is_collision(nxt): return candidate
            return 0
        cur_idx    = DIRECTIONS.index(env.direction)
        target_idx = DIRECTIONS.index(target_dir)
        diff       = (target_idx - cur_idx) % 4
        self._step = next_idx
        if diff == 0:   return 0
        elif diff == 1: return 1
        elif diff == 3: return 2
        else:           return 1


def evaluate_hamiltonian(size, episodes=50):
    assert size % 2 == 0, "Hamiltonian only valid for even grid sizes"
    max_steps = size * size * MAX_STEPS_K
    agent     = HamiltonianAgent(size)
    scores    = []
    for _ in range(episodes):
        env = SnakeEnv(size, obstacle_count=0, rep='local')
        env.reset(); agent.reset(env)
        food = 0; steps = 0; done = False; trunc = False
        while not done and not trunc and steps < max_steps:
            action = agent.act(env)
            _, _, done, trunc, ate = env.step(action)
            food += int(ate); steps += 1
        scores.append(food / (steps + 1e-8))
    return float(np.mean(scores)), float(np.std(scores))



# ── Training loops ─────────────────────────────────────────────────────────────

def train_dqn_vec(agent, size, obs_count, rep,
                  total_steps=DQN_STEPS, n_envs=N_ENVS_DQN, verbose_every=0):
    vec_env = make_vec_env(n_envs, size, obs_count, rep)
    obs     = vec_env.reset()
    food_log = []; ep_food = np.zeros(n_envs, dtype=np.int32); step = 0
    while step < total_steps:
        actions = agent.select_actions(obs)
        next_obs, rews, dones, truncs, infos = vec_env.step(actions)
        agent.buffer.push_batch(obs, actions, rews, next_obs, dones)
        agent.learn()
        for i, info in enumerate(infos):
            if info['ate_food']: ep_food[i] += 1
            if info['done'] or info['trunc']:
                food_log.append(ep_food[i]); ep_food[i] = 0
        obs = next_obs; step += n_envs
        if verbose_every and step % verbose_every < n_envs:
            recent = np.mean(food_log[-50:]) if food_log else 0.0
            print(f'  step {step:7d} | eps={agent.eps:.3f} | food/ep={recent:.2f}')
    vec_env.close()
    return food_log


def train_ppo_vec(agent, size, obs_count, rep,
                  total_steps=PPO_STEPS, verbose_every=0):
    vec_env = make_vec_env(agent.n_envs, size, obs_count, rep)
    obs     = vec_env.reset()
    food_log = []; step = 0; update = 0
    steps_per_update = agent.rollout_len * agent.n_envs
    while step < total_steps:
        last_obs, ep_food = agent.collect_rollout(vec_env, obs)
        last_dones  = np.zeros(agent.n_envs, dtype=bool)
        last_truncs = np.zeros(agent.n_envs, dtype=bool)
        adv, ret    = agent.compute_gae(last_obs, last_dones, last_truncs)
        agent.update(adv, ret)
        food_log.extend(ep_food)
        obs    = last_obs; step += steps_per_update; update += 1
        if verbose_every and update % verbose_every == 0:
            recent = np.mean(food_log[-50:]) if food_log else 0.0
            print(f'  update {update} | step {step:,} | food/ep={recent:.2f}')
    vec_env.close()
    return food_log


print('Hamiltonian + training loops defined.')

Hamiltonian + training loops defined.


In [9]:
# ── Cell 7: Evaluation helpers ────────────────────────────────────────────────

def _food_per_step(agent, size, obs_count, rep, episodes, is_ppo=False, is_ham=False):
    max_steps = size * size * MAX_STEPS_K
    scores    = []
    for _ in range(episodes):
        env   = SnakeEnv(size, obs_count, rep)
        state = env.reset()
        food  = 0; steps = 0
        if is_ham: agent.reset(env)
        done = False; trunc = False
        while not done and not trunc and steps < max_steps:
            if is_ham:
                action = agent.act(env)
            elif is_ppo:
                with torch.no_grad():
                    t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
                    action = agent.ac.act_greedy(t).item()
            else:
                action = agent.select_actions(state[None], greedy=True)[0]
            state, _, done, trunc, ate = env.step(action)
            food += int(ate); steps += 1
        scores.append(food / (steps + 1e-8))
    return scores


def evaluate_agent(agent, size, obs_count, rep,
                   episodes=EVAL_EPISODES, is_ppo=False, is_ham=False):
    scores = _food_per_step(agent, size, obs_count, rep,
                             episodes, is_ppo=is_ppo, is_ham=is_ham)
    n  = len(scores)
    m  = float(np.mean(scores))
    ci = 1.96 * float(np.std(scores)) / math.sqrt(n)
    return m, ci


def random_agent_score(size, obs_count, rep, episodes=EVAL_EPISODES):
    max_steps = size * size * MAX_STEPS_K
    scores    = []
    for _ in range(episodes):
        env  = SnakeEnv(size, obs_count, rep)
        env.reset()
        food = 0; steps = 0; done = False; trunc = False
        while not done and not trunc and steps < max_steps:
            _, _, done, trunc, ate = env.step(random.randint(0, 2))
            food += int(ate); steps += 1
        scores.append(food / (steps + 1e-8))
    n = len(scores)
    return float(np.mean(scores)), 1.96 * float(np.std(scores)) / math.sqrt(n)


def multi_seed_eval(build_fn, train_fn, eval_cfgs, rep,
                    seeds=SEEDS, is_ppo=False, label='', eval_ep=30):
    """
    Trains `seeds` agents, evaluates each on eval_cfgs.
    Returns {cfg: [seed_mean_scores]}, {cfg: [all_episode_scores]}.
    Also returns training food_logs per seed.
    """
    results   = {cfg: [] for cfg in eval_cfgs}
    all_scores= {cfg: [] for cfg in eval_cfgs}
    train_logs = []

    for seed in range(seeds):
        torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
        agent = build_fn()
        log   = train_fn(agent)
        train_logs.append(log)
        for cfg in eval_cfgs:
            size, obs = cfg
            ep_scores = _food_per_step(agent, size, obs, rep, eval_ep, is_ppo=is_ppo)
            results[cfg].append(float(np.mean(ep_scores)))
            all_scores[cfg].extend(ep_scores)
        print(f'  seed {seed+1}/{seeds} done [{label}]')

    return results, all_scores, train_logs


print('Evaluation helpers defined.')

Evaluation helpers defined.


In [10]:
# ── Cell 8: Plotting utilities ────────────────────────────────────────────────

PALETTE = {
    # models
    'dqn':        '#4ECDC4',
    'doubleDQN':  '#FF6B6B',
    'doubledqn':  '#FF6B6B',
    'ppo':        '#FFE66D',
    # reps
    'local':      '#F4A261',
    'extended':   '#A8DADC',
    'hybrid':     '#2A9D8F',
    # baselines
    'hamiltonian':'#C77DFF',
    'random':     '#6D6D6D',
}

MODEL_COLORS = {'DQN': '#4ECDC4', 'DoubleDQN': '#FF6B6B', 'PPO': '#FFE66D'}
REP_COLORS   = {'local': '#F4A261', 'extended': '#A8DADC', 'hybrid': '#2A9D8F'}
LINE_STYLES  = ['-', '--', '-.']

BG   = '#0D1117'
GRID_CLR = '#1E2A38'
TICK_CLR = '#AAAAAA'

def _save(fig, name):
    os.makedirs('snake_results', exist_ok=True)
    # sanitise filename
    safe = name.replace('/', '_').replace('–', '').replace(':', '').replace('×', 'x')
    path = f'/content/drive/MyDrive/snake_results/{safe}.png'
    fig.savefig(path, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f'  → saved {path}')

def _dark_fig(nrows=1, ncols=1, **kwargs):
    fig, axes = plt.subplots(nrows, ncols, **kwargs)
    fig.patch.set_facecolor(BG)
    ax_list = np.array(axes).flatten() if isinstance(axes, (np.ndarray, list)) else [axes]
    for ax in ax_list:
        ax.set_facecolor(BG)
        ax.grid(True, color=GRID_CLR, linewidth=0.6, alpha=0.8)
        for sp in ax.spines.values(): sp.set_edgecolor('#333333')
        ax.tick_params(colors=TICK_CLR)
    return fig, axes

def smooth(x, w=30):
    x = np.array(x)
    if len(x) < w: return x
    return np.convolve(x, np.ones(w) / w, mode='valid')


def plot_line_with_ci(ax, x, means, cis, label, color, ls='-', alpha_band=0.18):
    """Plot a line with shaded CI band."""
    means = np.array(means); cis = np.array(cis)
    ax.plot(x, means, color=color, linewidth=2.2, linestyle=ls, label=label)
    ax.fill_between(x, means - cis, means + cis, color=color, alpha=alpha_band)


def _label_axes(ax, title, xlabel, ylabel, legend=True):
    ax.set_title(title, color='white', fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel(xlabel, color=TICK_CLR, fontsize=10)
    ax.set_ylabel(ylabel, color=TICK_CLR, fontsize=10)
    if legend:
        leg = ax.legend(framealpha=0.25, labelcolor='white', fontsize=9,
                        facecolor='#1A1A2E', edgecolor='#333333')


# ── convenience wrappers ───────────────────────────────────────────────────────

def ci_from_seeds(seed_means):
    """95% CI from a list of per-seed means."""
    n = len(seed_means)
    if n == 0: return 0.0
    return 1.96 * float(np.std(seed_means)) / math.sqrt(n)


def results_to_mean_ci(results, cfgs):
    """results: {cfg: [seed_means]}  → arrays of means and CIs over cfgs."""
    means = [np.mean(results.get(c, [0])) for c in cfgs]
    cis   = [ci_from_seeds(results.get(c, [0])) for c in cfgs]
    return np.array(means), np.array(cis)


print('Plotting utilities defined.')

Plotting utilities defined.


---
## Experiment 0 — Baselines
Hamiltonian path agent + random agent across all grid sizes and obstacle configs.

In [11]:
# ── Cell 9: Baselines (cached) ─────────────────────────────────────────────────

def _run_baselines():
    print('  Computing Hamiltonian baseline ...')
    ham = {}
    for size in [s for s in GRID_SIZES if s % 2 == 0]:
        m, s = evaluate_hamiltonian(size, episodes=50)
        ham[size] = (m, s)
        print(f'    {size}x{size}: {m:.5f} ± {s:.5f}')

    print('  Computing Random baseline ...')
    rand_size = {}
    for cfg in EVAL_CFGS:
        m, ci = random_agent_score(*cfg, 'hybrid', episodes=50)
        rand_size[cfg] = (m, ci)

    rand_obs = {}
    for cfg in OBS_CFGS:
        m, ci = random_agent_score(*cfg, 'hybrid', episodes=50)
        rand_obs[cfg] = (m, ci)

    return {'ham': ham, 'rand_size': rand_size, 'rand_obs': rand_obs}

baselines = cached('baselines', _run_baselines)
ham_scores   = baselines['ham']
rand_size    = baselines['rand_size']
rand_obs     = baselines['rand_obs']
print('\nHamiltonian scores:')
for k, v in ham_scores.items():
    print(f'  {k}x{k}: {v[0]:.5f} ± {v[1]:.5f}')

  [SKIP] "baselines" already cached.

Hamiltonian scores:
  6x6: 0.05907 ± 0.01397
  8x8: 0.02990 ± 0.00864
  10x10: 0.01847 ± 0.00396
  12x12: 0.01370 ± 0.00343
  14x14: 0.00969 ± 0.00248


---
## Experiment A — Representation Ablation (DoubleDQN, 9×9)
Train DoubleDQN with local / extended / hybrid features, evaluate on all grid sizes.

In [12]:
# ── Cell 10: Exp A — Representation Ablation ──────────────────────────────────

def _run_rep_ablation():
    rep_results = {}   # {rep: {cfg: [seed_means]}}
    rep_logs    = {}   # {rep: [[food_log_seed0], [food_log_seed1], ...]}

    for rep in REPS:
        dim = SnakeEnv.STATE_DIMS[rep]
        print(f'\n  rep={rep} ({dim}-dim)')

        def make(d=dim):  return DQNAgent(d, double=True)
        def train(a, r=rep): return train_dqn_vec(a, TRAIN_SIZE, TRAIN_OBS, r)

        res, _, logs = multi_seed_eval(make, train, EVAL_CFGS, rep, label=f'DDQN-{rep}')
        rep_results[rep] = res
        rep_logs[rep]    = logs

    return {'results': rep_results, 'logs': rep_logs}

exp_a = cached('exp_a_rep_ablation', _run_rep_ablation)
rep_results = exp_a['results']
rep_logs    = exp_a['logs']
print('Experiment A loaded.')

  [SKIP] "exp_a_rep_ablation" already cached.
Experiment A loaded.


---
## Experiment B — Agent Comparison (hybrid, 9×9)
Train DQN / DoubleDQN / PPO with hybrid features. Evaluate on all grid sizes.

In [13]:
# ── Cell 11: Exp B — Agent Comparison ─────────────────────────────────────────

REP = 'hybrid'
S_DIM = SnakeEnv.STATE_DIMS[REP]

def make_dqn():   return DQNAgent(S_DIM, double=False)
def make_ddqn():  return DQNAgent(S_DIM, double=True)
def make_ppo():   return PPOAgent(S_DIM, n_actions=3, n_envs=N_ENVS_PPO)

def train_dqn_fixed(a):  return train_dqn_vec(a, TRAIN_SIZE, TRAIN_OBS, REP)
def train_ppo_fixed(a):  return train_ppo_vec(a, TRAIN_SIZE, TRAIN_OBS, REP)

AGENTS_SPEC = [
    ('DQN',       make_dqn,  train_dqn_fixed, False),
    ('DoubleDQN', make_ddqn, train_dqn_fixed, False),
    ('PPO',       make_ppo,  train_ppo_fixed, True),
]

def _run_agent_comparison():
    agent_results = {}
    agent_logs    = {}
    for name, build_fn, train_fn, is_ppo in AGENTS_SPEC:
        print(f'\n  Agent = {name}')
        res, _, logs = multi_seed_eval(build_fn, train_fn, EVAL_CFGS, REP,
                                       is_ppo=is_ppo, label=name)
        agent_results[name] = res
        agent_logs[name]    = logs
    return {'results': agent_results, 'logs': agent_logs}

exp_b = cached('exp_b_agent_comparison', _run_agent_comparison)
agent_results = exp_b['results']
agent_logs    = exp_b['logs']
print('Experiment B loaded.')

  [SKIP] "exp_b_agent_comparison" already cached.
Experiment B loaded.


---
## Experiment C — Obstacle Generalization
All three agents (hybrid), trained on 9×9 no obstacles, evaluated on 9×9 with 0–8 obstacles.

In [14]:
# ── Cell 12: Exp C — Obstacle Generalization ───────────────────────────────────

def _run_obstacle_gen():
    obs_results = {}
    for name, build_fn, train_fn, is_ppo in AGENTS_SPEC:
        print(f'\n  {name} — obstacle transfer')
        res, _, _ = multi_seed_eval(build_fn, train_fn, OBS_CFGS, REP,
                                    is_ppo=is_ppo, label=name)
        obs_results[name] = res
    return obs_results

exp_c = cached('exp_c_obstacle_gen', _run_obstacle_gen)
obs_results = exp_c
print('Experiment C loaded.')

  [SKIP] "exp_c_obstacle_gen" already cached.
Experiment C loaded.


---
## Experiment D — All reps × All models
For the "fix rep, vary model" and "fix model, vary rep" line plots, we need the full 3×3 grid of (model, rep) evaluated on EVAL_CFGS and OBS_CFGS.

In [15]:
# ── Cell 13: Exp D — Full rep × model grid ────────────────────────────────────

def _run_full_grid():
    """
    grid[rep][model_name] = {
        'size': {cfg: [seed_means]},
        'obs':  {cfg: [seed_means]}
    }
    """
    grid = {}
    for rep in REPS:
        dim = SnakeEnv.STATE_DIMS[rep]
        grid[rep] = {}
        for model_name, _, _, is_ppo in AGENTS_SPEC:
            print(f'  rep={rep}  model={model_name}')
            if model_name == 'PPO':
                def make_m(d=dim): return PPOAgent(d, n_actions=3, n_envs=N_ENVS_PPO)
                def train_m(a, r=rep): return train_ppo_vec(a, TRAIN_SIZE, TRAIN_OBS, r)
            elif model_name == 'DoubleDQN':
                def make_m(d=dim): return DQNAgent(d, double=True)
                def train_m(a, r=rep): return train_dqn_vec(a, TRAIN_SIZE, TRAIN_OBS, r)
            else:  # DQN
                def make_m(d=dim): return DQNAgent(d, double=False)
                def train_m(a, r=rep): return train_dqn_vec(a, TRAIN_SIZE, TRAIN_OBS, r)

            res_size, _, _ = multi_seed_eval(make_m, train_m, EVAL_CFGS, rep,
                                             is_ppo=is_ppo, label=f'{model_name}/{rep}')
            res_obs, _, _  = multi_seed_eval(make_m, train_m, OBS_CFGS,  rep,
                                             is_ppo=is_ppo, label=f'{model_name}/{rep}/obs')
            grid[rep][model_name] = {'size': res_size, 'obs': res_obs}
    return grid

exp_d = cached('exp_d_full_grid', _run_full_grid)
full_grid = exp_d
print('Experiment D loaded.')

  [SKIP] "exp_d_full_grid" already cached.
Experiment D loaded.


---
## Plotting — All figures
All experiments are done. This cell block generates every figure from cached data only — safe to rerun without any training.

In [29]:
# ── Cell 14: Figure 1 — Training Curves (all three models, hybrid, 9×9) ────────
fig, ax = _dark_fig(figsize=(12, 5))

EP_CAP = 7000

for name, color in MODEL_COLORS.items():
    logs = agent_logs[name]
    min_len = min(min(len(l) for l in logs), EP_CAP)
    clipped  = np.array([l[:min_len] for l in logs])
    window   = max(20, min_len // 50)
    smoothed = np.array([smooth(row, window) for row in clipped])
    eps_x    = np.arange(smoothed.shape[1])
    mean_sm  = smoothed.mean(0)
    std_sm   = smoothed.std(0)
    ax.plot(eps_x, mean_sm, color=color, linewidth=2.0, label=name)
    ax.fill_between(eps_x, mean_sm - std_sm, mean_sm + std_sm, color=color, alpha=0.15)

_label_axes(ax, 'Training Curves — All Models (hybrid, 9×9)',
            'Episode', 'Food per Episode (smoothed)')
fig.tight_layout()
_save(fig, 'fig1_training_curves_all_models')

  → saved /content/drive/MyDrive/snake_results/fig1_training_curves_all_models.png


In [17]:
# ── Cell 15: Figure 2 — Performance on 9×9 (all three models, hybrid) ──────────

fig, ax1 = _dark_fig(1, 1, figsize=(7, 5))

model_names = list(MODEL_COLORS.keys())
x = np.arange(len(model_names))

means_fs, cis_fs = [], []
for name in model_names:
    vals = agent_results[name].get(TRAIN_CFG, [0])
    means_fs.append(np.mean(vals))
    cis_fs.append(ci_from_seeds(vals))

ax1.bar(x, means_fs, color=[MODEL_COLORS[n] for n in model_names],
        width=0.5, alpha=0.85,
        yerr=cis_fs, error_kw=dict(ecolor='white', capsize=5, alpha=0.7))

r_m = rand_size[TRAIN_CFG][0]
ax1.axhline(r_m, color=PALETTE['random'], linestyle=':', linewidth=1.2,
            label=f'Random ({r_m:.4f})')

ax1.set_xticks(x); ax1.set_xticklabels(model_names, color=TICK_CLR)
_label_axes(ax1, '9×9 Performance — Hybrid Features', 'Model', 'Food / Step')
fig.tight_layout()
_save(fig, 'fig2_9x9_model_comparison_hybrid')

  → saved /content/drive/MyDrive/snake_results/fig2_9x9_model_comparison_hybrid.png


In [18]:
# ── Cell 16: Figure 3 — Rep Ablation on 9×9 (per model, DoubleDQN used for ablation) ──

fig, (ax1, ax2) = _dark_fig(1, 2, figsize=(13, 5))

rep_names = REPS
x = np.arange(len(rep_names))
r_m = rand_size[TRAIN_CFG][0]

# ---- food/step ---------------------------------------------------------------
for i, rep in enumerate(rep_names):
    vals = rep_results[rep].get(TRAIN_CFG, [0])
    m = np.mean(vals); ci = ci_from_seeds(vals)
    ax1.bar(i, m, color=REP_COLORS[rep], width=0.5, alpha=0.85, label=rep,
            yerr=ci, error_kw=dict(ecolor='white', capsize=5, alpha=0.7))

ax1.axhline(r_m, color=PALETTE['random'], linestyle=':', linewidth=1.2, label='Random')
ax1.set_xticks(x); ax1.set_xticklabels(rep_names, color=TICK_CLR)
_label_axes(ax1, 'Representation Ablation on 9×9 (DoubleDQN)', 'Feature Space', 'Food / Step')

# ---- normalised to random ----------------------------------------------------
for i, rep in enumerate(rep_names):
    vals = rep_results[rep].get(TRAIN_CFG, [0])
    m = np.mean(vals) / (r_m + 1e-8); ci = ci_from_seeds(vals) / (r_m + 1e-8)
    ax2.bar(i, m, color=REP_COLORS[rep], width=0.5, alpha=0.85, label=rep,
            yerr=ci, error_kw=dict(ecolor='white', capsize=5, alpha=0.7))

ax2.axhline(1.0, color=PALETTE['random'], linestyle=':', linewidth=1.2, label='Random = 1')
ax2.set_xticks(x); ax2.set_xticklabels(rep_names, color=TICK_CLR)
_label_axes(ax2, 'Representation Ablation — Normalised to Random', 'Feature Space', 'Agent / Random')

fig.tight_layout()
_save(fig, 'fig3_rep_ablation_9x9_doubledqn')

  → saved /content/drive/MyDrive/snake_results/fig3_rep_ablation_9x9_doubledqn.png


In [19]:
# ── Cell 17: Figures 4a–4c — Size Generalisation (fix rep, 3 model lines each) ──
# 3 graphs (one per rep), each with lines for DQN / DoubleDQN / PPO
# 2 panels per graph: food/step  |  normalised
# only plot even sizes where Hamiltonian exists
# only plot even sizes where Hamiltonian exists
EVEN_CFGS = [(s, 0) for s in GRID_SIZES if s % 2 == 0]
x_sizes   = [s for s, _ in EVEN_CFGS]

for rep in REPS:
    fig, (ax1, ax2) = _dark_fig(1, 2, figsize=(14, 5))

    for model_name, color in MODEL_COLORS.items():
        res = full_grid[rep][model_name]['size']
        means, cis = results_to_mean_ci(res, EVEN_CFGS)  # <-- EVEN_CFGS not EVAL_CFGS
        plot_line_with_ci(ax1, x_sizes, means, cis, model_name, color)

        h_vals = np.array([ham_scores[s][0] for s in x_sizes])
        plot_line_with_ci(ax2, x_sizes, means / (h_vals + 1e-8),
                          cis / (h_vals + 1e-8), model_name, color)

    h_vals = np.array([ham_scores[s][0] for s in x_sizes])
    r_vals = np.array([rand_size[(s, 0)][0] for s in x_sizes])
    ax1.plot(x_sizes, h_vals, color=PALETTE['hamiltonian'], linestyle='--',
             linewidth=1.5, label='Hamiltonian')
    ax1.plot(x_sizes, r_vals, color=PALETTE['random'], linestyle=':',
             linewidth=1.2, label='Random')
    ax2.axhline(1.0, color=PALETTE['hamiltonian'], linestyle='--',
                linewidth=1.5, label='Hamiltonian = 1')

    ax1.set_xticks(x_sizes)
    ax2.set_xticks(x_sizes)
    _label_axes(ax1, f'Size Generalisation — {rep} features (food/step)',
                'Grid Size', 'Food / Step')
    _label_axes(ax2, f'Size Generalisation — {rep} features (normalised)',
                'Grid Size', 'Agent / Hamiltonian')
    fig.suptitle(f'Feature Space: {rep}', color='white', fontsize=13, y=1.01)
    fig.tight_layout()
    _save(fig, f'fig4_size_gen_{rep}')

  → saved /content/drive/MyDrive/snake_results/fig4_size_gen_local.png
  → saved /content/drive/MyDrive/snake_results/fig4_size_gen_extended.png
  → saved /content/drive/MyDrive/snake_results/fig4_size_gen_hybrid.png


In [20]:
# ── Cell 18: Figures 5a–5c — Obstacle Generalisation (fix rep, 3 model lines) ──

x_obs = [o for _, o in OBS_CFGS]

for rep in REPS:
    fig, (ax1, ax2) = _dark_fig(1, 2, figsize=(14, 5))
    r_ref = rand_size[TRAIN_CFG][0]  # 9×9 random reference (no Hamiltonian for odd grid)

    for model_name, color in MODEL_COLORS.items():
        res = full_grid[rep][model_name]['obs']
        means, cis = results_to_mean_ci(res, OBS_CFGS)
        plot_line_with_ci(ax1, x_obs, means, cis, model_name, color)
        plot_line_with_ci(ax2, x_obs, means / (r_ref + 1e-8),
                          cis / (r_ref + 1e-8), model_name, color)

    r_vals = np.array([rand_obs[(TRAIN_SIZE, o)][0] for o in x_obs])
    ax1.plot(x_obs, r_vals, color=PALETTE['random'], linestyle=':',
             linewidth=1.2, label='Random')
    ax2.axhline(1.0, color=PALETTE['random'], linestyle=':',
                linewidth=1.2, label='Random = 1')

    ax1.set_xticks(x_obs)
    ax2.set_xticks(x_obs)
    _label_axes(ax1, f'Obstacle Generalisation — {rep} features (food/step)',
                'Obstacle Count', 'Food / Step')
    _label_axes(ax2, f'Obstacle Generalisation — {rep} features (normalised to random)',
                'Obstacle Count', 'Agent / Random')
    fig.suptitle(f'Feature Space: {rep}', color='white', fontsize=13, y=1.01)
    fig.tight_layout()
    _save(fig, f'fig5_obs_gen_{rep}')

  → saved /content/drive/MyDrive/snake_results/fig5_obs_gen_local.png
  → saved /content/drive/MyDrive/snake_results/fig5_obs_gen_extended.png
  → saved /content/drive/MyDrive/snake_results/fig5_obs_gen_hybrid.png


In [21]:
# ── Cell 19: Figure 6 — Fix Model, Vary Rep (size generalisation) ──────────────

EVEN_CFGS = [(s, 0) for s in GRID_SIZES if s % 2 == 0]
x_sizes   = [s for s, _ in EVEN_CFGS]
fig, axes = _dark_fig(1, 3, figsize=(18, 5))

for ax, (model_name, _, _, _) in zip(axes, AGENTS_SPEC):
    for rep, ls in zip(REPS, LINE_STYLES):
        color = REP_COLORS[rep]
        res   = full_grid[rep][model_name]['size']
        means, cis = results_to_mean_ci(res, EVEN_CFGS)
        plot_line_with_ci(ax, x_sizes, means, cis, rep, color, ls=ls)

    h_vals = np.array([ham_scores[s][0] for s in x_sizes])
    ax.plot(x_sizes, h_vals, color=PALETTE['hamiltonian'], linestyle='--',
            linewidth=1.4, label='Hamiltonian')
    ax.set_xticks(x_sizes)
    _label_axes(ax, f'{model_name} — Feature Space Comparison',
                'Grid Size', 'Food / Step')

fig.suptitle('Fix Model → Effect of Feature Representation on Size Generalisation',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig6_fix_model_vary_rep_size')

  → saved /content/drive/MyDrive/snake_results/fig6_fix_model_vary_rep_size.png


In [22]:
# ── Cell 20: Figure 7 — Fix Rep, Vary Model (size generalisation) ──────────────
# 3 subplots (one per rep), 3 model lines each

fig, axes = _dark_fig(1, 3, figsize=(18, 5))

for ax, rep in zip(axes, REPS):
    for model_name, ls in zip(MODEL_COLORS.keys(), LINE_STYLES):
        color = MODEL_COLORS[model_name]
        res   = full_grid[rep][model_name]['size']
        means, cis = results_to_mean_ci(res, EVEN_CFGS)
        plot_line_with_ci(ax, x_sizes, means, cis, model_name, color, ls=ls)

    h_vals = np.array([ham_scores[s][0] for s in x_sizes])
    ax.plot(x_sizes, h_vals, color=PALETTE['hamiltonian'], linestyle='--',
            linewidth=1.4, label='Hamiltonian')
    ax.set_xticks(x_sizes)
    _label_axes(ax, f'{rep} features — Model Comparison', 'Grid Size', 'Food / Step')

fig.suptitle('Fix Feature Rep → Effect of Model on Size Generalisation',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig7_fix_rep_vary_model_size')

  → saved /content/drive/MyDrive/snake_results/fig7_fix_rep_vary_model_size.png


In [23]:
# ── Cell 21: Figure 8 — Fix Model, Vary Rep (obstacle generalisation) ──────────

x_obs = [o for _, o in OBS_CFGS]
fig, axes = _dark_fig(1, 3, figsize=(18, 5))

for ax, (model_name, _, _, _) in zip(axes, AGENTS_SPEC):
    for rep, ls in zip(REPS, LINE_STYLES):
        color = REP_COLORS[rep]
        res   = full_grid[rep][model_name]['obs']
        means, cis = results_to_mean_ci(res, OBS_CFGS)
        plot_line_with_ci(ax, x_obs, means, cis, rep, color, ls=ls)

    ax.set_xticks(x_obs)
    _label_axes(ax, f'{model_name} — Feature Space vs Obstacles',
                'Obstacle Count', 'Food / Step')

fig.suptitle('Fix Model → Effect of Feature Rep on Obstacle Generalisation',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig8_fix_model_vary_rep_obs')

  → saved /content/drive/MyDrive/snake_results/fig8_fix_model_vary_rep_obs.png


In [24]:
# ── Cell 22: Figure 9 — Fix Rep, Vary Model (obstacle generalisation) ──────────

fig, axes = _dark_fig(1, 3, figsize=(18, 5))

for ax, rep in zip(axes, REPS):
    for model_name, ls in zip(MODEL_COLORS.keys(), LINE_STYLES):
        color = MODEL_COLORS[model_name]
        res   = full_grid[rep][model_name]['obs']
        means, cis = results_to_mean_ci(res, OBS_CFGS)
        plot_line_with_ci(ax, x_obs, means, cis, model_name, color, ls=ls)

    ax.set_xticks(x_obs)
    _label_axes(ax, f'{rep} features — Model vs Obstacles',
                'Obstacle Count', 'Food / Step')

fig.suptitle('Fix Feature Rep → Effect of Model on Obstacle Generalisation',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig9_fix_rep_vary_model_obs')

  → saved /content/drive/MyDrive/snake_results/fig9_fix_rep_vary_model_obs.png


In [25]:
# ── Cell 23: Figure 10 — Summary Stats Table Plot ─────────────────────────────
# For each model (hybrid), show: mean ± CI for each grid size
# Violin-style: plot distribution of per-seed means + CI

fig, axes = _dark_fig(1, 3, figsize=(18, 5))

x_sizes = [s for s, _ in EVEN_CFGS]

for ax, (model_name, color) in zip(axes, MODEL_COLORS.items()):
    res = agent_results[model_name]   # {cfg: [seed_means]}

    # Box-like: per-size seed distribution
    all_seed_vals = [res.get(cfg, [0]) for cfg in EVEN_CFGS]
    positions = np.arange(len(EVEN_CFGS))

    bp = ax.boxplot(all_seed_vals, positions=positions,
                    widths=0.35, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.5),
                    medianprops=dict(color='white', linewidth=2),
                    whiskerprops=dict(color=TICK_CLR),
                    capprops=dict(color=TICK_CLR),
                    flierprops=dict(marker='o', color=color, alpha=0.5, markersize=4))

    # Overlay mean line
    means = [np.mean(v) for v in all_seed_vals]
    ax.plot(positions, means, color=color, linewidth=2.0, marker='D',
            markersize=6, label='Seed mean')

    h_vals = [ham_scores[s][0] for s in x_sizes]
    ax.plot(positions, h_vals, color=PALETTE['hamiltonian'], linestyle='--',
            linewidth=1.4, label='Hamiltonian')

    ax.set_xticks(positions)
    ax.set_xticklabels([f'{s}×{s}' for s in x_sizes], color=TICK_CLR, fontsize=9)
    _label_axes(ax, f'{model_name} — Seed Distribution by Grid Size',
                'Grid Size', 'Food / Step')

fig.suptitle('Seed Variance Across Grid Sizes (hybrid, 3 seeds)',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig10_seed_variance_by_size')

  → saved /content/drive/MyDrive/snake_results/fig10_seed_variance_by_size.png


In [31]:
# ── Cell 24: Figure 11 — Generalisation Gap Heatmap ──────────────────────────
# Rows: models. Cols: eval configs (sizes). Colour: train_score - test_score
fig, axes = _dark_fig(1, 2, figsize=(15, 5))
model_names = list(MODEL_COLORS.keys())
sizes_only  = [s for s, _ in EVEN_CFGS]
obs_only    = [o for _, o in OBS_CFGS]
# ---- Size generalisation gap -------------------------------------------------
gap_mat_size = np.zeros((len(model_names), len(EVEN_CFGS)))
for ri, name in enumerate(model_names):
    train_m = np.mean(agent_results[name].get(TRAIN_CFG, [0]))
    for ci, cfg in enumerate(EVEN_CFGS):
        test_m = np.mean(agent_results[name].get(cfg, [0]))
        gap_mat_size[ri, ci] = train_m - test_m
im1 = axes[0].imshow(gap_mat_size, aspect='auto', cmap='RdYlGn',
                      vmin=-0.02, vmax=0.05)
fig.colorbar(im1, ax=axes[0]).set_label('Train − Test (food/step)', color=TICK_CLR)
axes[0].set_xticks(range(len(EVEN_CFGS)))
axes[0].set_xticklabels([f'{s}×{s}' for s in sizes_only], color=TICK_CLR)
axes[0].set_yticks(range(len(model_names)))
axes[0].set_yticklabels(model_names, color=TICK_CLR)
for ri in range(len(model_names)):
    for ci in range(len(EVEN_CFGS)):
        axes[0].text(ci, ri, f'{gap_mat_size[ri,ci]:.4f}',
                     ha='center', va='center', color='white', fontsize=8)
_label_axes(axes[0], 'Generalisation Gap — Grid Size', 'Grid Size', 'Model', legend=False)
# ---- Obstacle generalisation gap ---------------------------------------------
gap_mat_obs = np.zeros((len(model_names), len(OBS_CFGS)))
for ri, name in enumerate(model_names):
    train_m = np.mean(obs_results[name].get(TRAIN_CFG, [0]))
    for ci, cfg in enumerate(OBS_CFGS):
        test_m = np.mean(obs_results[name].get(cfg, [0]))
        gap_mat_obs[ri, ci] = train_m - test_m
im2 = axes[1].imshow(gap_mat_obs, aspect='auto', cmap='RdYlGn_r',
                      vmin=-0.02, vmax=0.05)
fig.colorbar(im2, ax=axes[1]).set_label('Train − Test (food/step)', color=TICK_CLR)
axes[1].set_xticks(range(len(OBS_CFGS)))
axes[1].set_xticklabels([f'obs={o}' for o in obs_only], color=TICK_CLR)
axes[1].set_yticks(range(len(model_names)))
axes[1].set_yticklabels(model_names, color=TICK_CLR)
for ri in range(len(model_names)):
    for ci in range(len(OBS_CFGS)):
        axes[1].text(ci, ri, f'{gap_mat_obs[ri,ci]:.4f}',
                     ha='center', va='center', color='white', fontsize=8)
_label_axes(axes[1], 'Generalisation Gap — Obstacles', 'Obstacle Count', 'Model', legend=False)
fig.suptitle('Generalisation Gap Heatmaps (train=9×9, no obstacles)',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig11_generalisation_gap_heatmaps')

  → saved /content/drive/MyDrive/snake_results/fig11_generalisation_gap_heatmaps.png


In [27]:
# ── Cell 25: Figure 12 — Normalised Size Generalisation (all reps, all models) ──
# Companion to figs 4a-c but showing normalised metric for both fix-rep and fix-model

x_sizes = [s for s, _ in EVEN_CFGS]

# Fix rep, vary model — normalised
fig, axes = _dark_fig(1, 3, figsize=(18, 5))
for ax, rep in zip(axes, REPS):
    for model_name, ls in zip(MODEL_COLORS.keys(), LINE_STYLES):
        color  = MODEL_COLORS[model_name]
        res    = full_grid[rep][model_name]['size']
        means, cis = results_to_mean_ci(res, EVEN_CFGS)
        h_vals = np.array([ham_scores[s][0] for s in x_sizes])
        plot_line_with_ci(ax, x_sizes, means / (h_vals + 1e-8),
                          cis / (h_vals + 1e-8), model_name, color, ls=ls)
    ax.axhline(1.0, color=PALETTE['hamiltonian'], linestyle='--',
               linewidth=1.4, label='Hamiltonian = 1')
    ax.set_xticks(x_sizes)
    _label_axes(ax, f'{rep} — Normalised Size Gen', 'Grid Size', 'Agent / Hamiltonian')

fig.suptitle('Fix Feature Rep → Normalised Size Generalisation',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig12_norm_size_gen_fix_rep')

# Fix model, vary rep — normalised
fig, axes = _dark_fig(1, 3, figsize=(18, 5))
for ax, (model_name, _, _, _) in zip(axes, AGENTS_SPEC):
    for rep, ls in zip(REPS, LINE_STYLES):
        color  = REP_COLORS[rep]
        res    = full_grid[rep][model_name]['size']
        means, cis = results_to_mean_ci(res, EVEN_CFGS)
        h_vals = np.array([ham_scores[s][0] for s in x_sizes])
        plot_line_with_ci(ax, x_sizes, means / (h_vals + 1e-8),
                          cis / (h_vals + 1e-8), rep, color, ls=ls)
    ax.axhline(1.0, color=PALETTE['hamiltonian'], linestyle='--',
               linewidth=1.4, label='Hamiltonian = 1')
    ax.set_xticks(x_sizes)
    _label_axes(ax, f'{model_name} — Normalised Size Gen', 'Grid Size', 'Agent / Hamiltonian')

fig.suptitle('Fix Model → Normalised Size Generalisation',
             color='white', fontsize=13, y=1.02)
fig.tight_layout()
_save(fig, 'fig13_norm_size_gen_fix_model')

  → saved /content/drive/MyDrive/snake_results/fig12_norm_size_gen_fix_rep.png
  → saved /content/drive/MyDrive/snake_results/fig13_norm_size_gen_fix_model.png


In [28]:
# ── Cell 26: Print summary table ──────────────────────────────────────────────

print('=' * 80)
print('  SUMMARY — food/step (hybrid, train 9×9)')
print('=' * 80)
print(f'{"Config":>12} | {"Random":>14} | {"Hamiltonian":>14}', end='')
for name in MODEL_COLORS: print(f' | {name:>14}', end='')
print()
print('-' * 80)

for cfg in EVEN_CFGS:
    rm, rci = rand_size.get(cfg, (0, 0))
    hm, hci = ham_scores.get(cfg[0], (0, 0))
    row = f'{str(cfg):>12} | {rm:.4f}±{rci:.4f} | {hm:.4f}±{hci:.4f}'
    for name in MODEL_COLORS:
        vals = agent_results[name].get(cfg, [0])
        m  = np.mean(vals)
        ci = ci_from_seeds(vals)
        row += f' | {m:.4f}±{ci:.4f}'
    print(row)

print()
print(f'All plots saved to ./snake_results/')
print('=' * 80)

  SUMMARY — food/step (hybrid, train 9×9)
      Config |         Random |    Hamiltonian |            DQN |      DoubleDQN |            PPO
--------------------------------------------------------------------------------
      (6, 0) | 0.0246±0.0183 | 0.0591±0.0140 | 0.2149±0.0061 | 0.2200±0.0082 | 0.2232±0.0092
      (8, 0) | 0.0213±0.0122 | 0.0299±0.0086 | 0.1582±0.0033 | 0.1619±0.0021 | 0.1707±0.0121
     (10, 0) | 0.0054±0.0047 | 0.0185±0.0040 | 0.1307±0.0036 | 0.1308±0.0035 | 0.1345±0.0069
     (12, 0) | 0.0080±0.0060 | 0.0137±0.0034 | 0.1120±0.0024 | 0.1102±0.0006 | 0.1167±0.0023

All plots saved to ./snake_results/
